# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [50]:
%pip install requests pandas bs4

  Using cached bs4-0.0.2-py2.py3-none-any.whl.metadata (411 bytes)
Using cached bs4-0.0.2-py2.py3-none-any.whl (1.2 kB)
Note: you may need to restart the kernel to use updated packages.


## 1. Importación de librerías y configuración

In [51]:
import re
import time
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup

URL_BASE = "https://books.toscrape.com/catalogue/page-{}.html"

# Las valoraciones vienen como texto en la clase CSS (ej: "star-rating Three")
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

## 2. Exploración de la web
Antes de programar el scraping, descargamos la primera página del catálogo y comprobamos
cuántos libros muestra y cómo es el HTML de cada uno.

In [52]:
%pip install --upgrade --force-reinstall certifi
respuesta.encoding = "utf-8"
print("Código de estado:", respuesta.status_code)

soup = BeautifulSoup(respuesta.text, "html.parser")
articulos = soup.select("article.product_pod")
print("Libros en la página 1:", len(articulos))

# Estructura HTML de un libro del listado
print(articulos[0].prettify())

  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
Using cached certifi-2026.10.23-py3-none-any.whl (135 kB)
  Attempting uninstall: certifi
    Found existing installation: certifi 2026.10.23
    Uninstalling certifi-2026.10.23:
      Successfully uninstalled certifi-2026.10.23
Note: you may need to restart the kernel to use updated packages.
Código de estado: 200
Libros en la página 1: 20
<article class="product_pod">
 <div class="image_container">
  <a href="a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
 

## 3. Función auxiliar para descargar páginas
Creamos una función que descarga una URL y devuelve su `BeautifulSoup`.
Si la página no existe (error 404) devuelve `None`, lo que nos servirá para detectar el final del catálogo.

In [53]:
def obtener_soup(sesion, url):
    """Descarga una URL y devuelve el BeautifulSoup (o None si no existe)."""
    respuesta = sesion.get(url, timeout=15)
    if respuesta.status_code == 404:
        return None
    respuesta.raise_for_status()
    respuesta.encoding = "utf-8"  # evita que el símbolo £ salga como "Â£"
    return BeautifulSoup(respuesta.text, "html.parser")

## 4. Extracción de los datos del listado
En el listado de cada página tenemos el **título**, el **precio** y la **valoración**.
Probamos la extracción con el primer libro antes de generalizarla.

In [54]:
sesion = requests.Session()
sesion.headers.update({"User-Agent": "Mozilla/5.0 (lab de web scraping)"})

# Ignore a stale REQUESTS_CA_BUNDLE/CURL_CA_BUNDLE setting, if present.
sesion.trust_env = False
soup = obtener_soup(sesion, URL_BASE.format(1))
primer_libro = soup.select("article.product_pod")[0]

# Título
titulo = primer_libro.h3.a["title"]

# Precio (limpiamos cualquier carácter que no sea número o punto)
precio_texto = primer_libro.select_one("p.price_color").get_text(strip=True)
precio = float(re.sub(r"[^\d.]", "", precio_texto))

# Valoración
clases = primer_libro.select_one("p.star-rating")["class"]
rating = next((RATING_MAP[c] for c in clases if c in RATING_MAP), 0)

# Enlace a la página de detalle
url_libro = urljoin(URL_BASE.format(1), primer_libro.h3.a["href"])

print("Título:", titulo)
print("Precio (£):", precio)
print("Rating:", rating)
print("URL:", url_libro)

Título: A Light in the Attic
Precio (£): 51.77
Rating: 3
URL: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


## 5. Extracción de los datos de la página de detalle
El **UPC**, la **disponibilidad**, el **género** y la **descripción** solo aparecen en la página de cada libro,
así que creamos una función que entra en ella y los extrae.

In [55]:
def extraer_detalle(sesion, url_libro):
    """Entra en la página de un libro y extrae UPC, género, disponibilidad y descripción."""
    soup = obtener_soup(sesion, url_libro)

    # Tabla de información del producto (UPC, disponibilidad, etc.)
    tabla = {}
    for fila in soup.select("table.table-striped tr"):
        clave = fila.find("th").get_text(strip=True)
        valor = fila.find("td").get_text(strip=True)
        tabla[clave] = valor

    # Género: tercer elemento del breadcrumb (Home > Books > Género > Título)
    migas = soup.select("ul.breadcrumb li")
    genero = migas[2].get_text(strip=True) if len(migas) > 2 else None

    # Disponibilidad
    texto_disp = tabla.get("Availability", "")
    disponibilidad = "En stock" if "In stock" in texto_disp else "Sin stock"

    # Descripción (el párrafo que sigue al bloque #product_description)
    bloque_desc = soup.find("div", id="product_description")
    if bloque_desc:
        parrafo = bloque_desc.find_next_sibling("p")
        descripcion = parrafo.get_text(strip=True) if parrafo else None
    else:
        descripcion = None

    return {
        "UPC": tabla.get("UPC"),
        "Género": genero,
        "Disponibilidad": disponibilidad,
        "Descripción": descripcion,
    }

In [56]:
# Prueba con el libro que extrajimos antes
detalle = extraer_detalle(sesion, url_libro)
for clave, valor in detalle.items():
    print(f"{clave}: {valor}")

UPC: a897fe39b1053632
Género: Poetry
Disponibilidad: En stock
Descripción: It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,And your cradle, too?

## 6. Función final `scrape_books`
Unimos todo el proceso: recorremos todas las páginas del catálogo, **filtramos por valoración y precio
en el listado** (para no hacer peticiones innecesarias) y solo entramos al detalle de los libros que cumplen el filtro.
El resultado se devuelve como un DataFrame de `pandas`.

In [57]:
def scrape_books(min_rating, max_price):
    """
    Hace scraping de Books to Scrape y devuelve un DataFrame con los libros
    cuya valoración sea >= min_rating y cuyo precio sea <= max_price (en £).
    """
    sesion = requests.Session()
    sesion.headers.update({"User-Agent": "Mozilla/5.0 (lab de web scraping)"})

    libros = []
    pagina = 1

    while True:
        url_pagina = URL_BASE.format(pagina)
        soup = obtener_soup(sesion, url_pagina)

        # Cuando la página no existe (404) hemos recorrido todo el catálogo
        if soup is None:
            break

        for articulo in soup.select("article.product_pod"):
            # Título
            titulo = articulo.h3.a["title"]

            # Precio
            precio_texto = articulo.select_one("p.price_color").get_text(strip=True)
            precio = float(re.sub(r"[^\d.]", "", precio_texto))

            # Valoración
            clases = articulo.select_one("p.star-rating")["class"]
            rating = next((RATING_MAP[c] for c in clases if c in RATING_MAP), 0)

            # Filtramos ANTES de entrar al detalle para ahorrar peticiones
            if rating < min_rating or precio > max_price:
                continue

            # Datos de la página de detalle
            url_libro = urljoin(url_pagina, articulo.h3.a["href"])
            detalle = extraer_detalle(sesion, url_libro)

            libros.append({
                "UPC": detalle["UPC"],
                "Title": titulo,
                "Price (£)": precio,
                "Rating": rating,
                "Genre": detalle["Género"],
                "Availability": detalle["Disponibilidad"],
                "Description": detalle["Descripción"],
            })

        print(f"Página {pagina} procesada | libros acumulados: {len(libros)}")
        pagina += 1
        time.sleep(0.2)  # pausa corta para no saturar el servidor

    columnas = ["UPC", "Title", "Price (£)", "Rating", "Genre", "Availability", "Description"]
    return pd.DataFrame(libros, columns=columnas)

## 7. Ejecución del lab
Libros con valoración mínima de **4.0** y precio máximo de **£20**.

In [58]:
df_libros = scrape_books(min_rating=4.0, max_price=20)

print(f"\nTotal de libros encontrados: {len(df_libros)}")
df_libros.head(10)

Página 1 procesada | libros acumulados: 1
Página 2 procesada | libros acumulados: 3
Página 3 procesada | libros acumulados: 6
Página 4 procesada | libros acumulados: 6
Página 5 procesada | libros acumulados: 10
Página 6 procesada | libros acumulados: 12
Página 7 procesada | libros acumulados: 14
Página 8 procesada | libros acumulados: 14
Página 9 procesada | libros acumulados: 16
Página 10 procesada | libros acumulados: 16
Página 11 procesada | libros acumulados: 19
Página 12 procesada | libros acumulados: 20
Página 13 procesada | libros acumulados: 20
Página 14 procesada | libros acumulados: 22
Página 15 procesada | libros acumulados: 25
Página 16 procesada | libros acumulados: 29
Página 17 procesada | libros acumulados: 30
Página 18 procesada | libros acumulados: 30
Página 19 procesada | libros acumulados: 32
Página 20 procesada | libros acumulados: 33
Página 21 procesada | libros acumulados: 35
Página 22 procesada | libros acumulados: 37
Página 23 procesada | libros acumulados: 37
P

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,En stock,Aaron Ledbetter’s future had been planned out ...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,En stock,"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,En stock,A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,En stock,"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,En stock,"Every summer, Rose goes with her mom and dad t..."
5,709822d0b5bcb7f4,Thirst,17.27,5,Fiction,En stock,"On a searing summer Friday, Eddie Chapman has ..."
6,0fa6dceead7ce47a,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5,Sequential Art,En stock,THE LONG-AWAITED STORY OF FANGIRLS TAKING ON T...
7,0e691eda369f4e09,Princess Between Worlds (Wide-Awake Princess #5),13.34,5,Fantasy,En stock,Just as Annie and Liam are busy making plans t...
8,889139b8e9c4cb36,"Outcast, Vol. 1: A Darkness Surrounds Him (Out...",15.44,4,Sequential Art,En stock,NEW HORROR SERIES FROM THE WALKING DEAD CREATO...
9,72f9d5be3472d34e,Mama Tried: Traditional Italian Cooking for th...,14.02,4,Food and Drink,En stock,Cecilia Granata grew up cooking with her famil...


## 8. Revisión de los resultados
Comprobamos la estructura del DataFrame y que se cumplen los filtros.

In [59]:
df_libros.info()
print("\nRating mínimo:", df_libros["Rating"].min())
print("Precio máximo (£):", df_libros["Price (£)"].max())
print("\nLibros por género (top 10):")
print(df_libros["Genre"].value_counts().head(10))

<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UPC           75 non-null     str    
 1   Title         75 non-null     str    
 2   Price (£)     75 non-null     float64
 3   Rating        75 non-null     int64  
 4   Genre         75 non-null     str    
 5   Availability  75 non-null     str    
 6   Description   75 non-null     str    
dtypes: float64(1), int64(1), str(5)
memory usage: 4.2 KB

Rating mínimo: 4
Precio máximo (£): 19.69

Libros por género (top 10):
Genre
Default           10
Sequential Art     9
Young Adult        7
Nonfiction         6
Fiction            5
Poetry             4
Food and Drink     4
Fantasy            3
Romance            3
Philosophy         2
Name: count, dtype: int64


## 9. Pruebas con otros parámetros
Comprobamos que la función es versátil probando otros valores.

In [60]:
df_prueba = scrape_books(min_rating=5, max_price=10)
print(f"\nLibros con 5 estrellas y precio <= £10: {len(df_prueba)}")
df_prueba.head()

Página 1 procesada | libros acumulados: 0
Página 2 procesada | libros acumulados: 0
Página 3 procesada | libros acumulados: 0
Página 4 procesada | libros acumulados: 0
Página 5 procesada | libros acumulados: 0
Página 6 procesada | libros acumulados: 0
Página 7 procesada | libros acumulados: 0
Página 8 procesada | libros acumulados: 0
Página 9 procesada | libros acumulados: 0
Página 10 procesada | libros acumulados: 0
Página 11 procesada | libros acumulados: 0
Página 12 procesada | libros acumulados: 0
Página 13 procesada | libros acumulados: 0
Página 14 procesada | libros acumulados: 0
Página 15 procesada | libros acumulados: 0
Página 16 procesada | libros acumulados: 0
Página 17 procesada | libros acumulados: 0
Página 18 procesada | libros acumulados: 0
Página 19 procesada | libros acumulados: 0
Página 20 procesada | libros acumulados: 0
Página 21 procesada | libros acumulados: 0
Página 22 procesada | libros acumulados: 0
Página 23 procesada | libros acumulados: 0
Página 24 procesada 

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,f36d24c309e87e5b,An Abundance of Katherines,10.0,5,Young Adult,En stock,Katherine V thought boys were gross Katherine ...
